# 02 — Supervised Fine-Tuning (SFT) for German→English Translation

**What this notebook does and why it exists**

Supervised fine-tuning (SFT) is the first adaptation step: we show the model thousands of (German input, English output) examples and update its weights to be better at this specific task. Think of it as the model reading a large bilingual textbook. After SFT, the model will produce decent translations; the subsequent GRPO notebook will then refine the *quality* of those translations using reinforcement learning.

SFT is a necessary warm-up before RL. Without it, the model's output distribution is too diffuse for a reward signal to guide it effectively — the model must already be close to the right answer space for RL to work well.

**Runtime note:** This notebook is designed for a Colab TPU runtime (v2 or v3). There are important compatibility caveats about quantisation on TPUs — see the Decision note below.

---
## Decision note: Quantisation on Colab TPUs

**The honest picture:** 4-bit quantisation (via bitsandbytes or GPTQ) is a GPU technique. TPU runtimes do not support bitsandbytes natively, because bitsandbytes is implemented in CUDA. As of late 2024, Unsloth also does not support TPU backends — it is CUDA-only.

**What this means for you:**
- On a **TPU runtime**, we train in `bfloat16` (the native TPU precision) without 4-bit quantisation. A 1.5B model in bfloat16 uses approximately 3 GB of memory, which fits comfortably on a v2-8 TPU (8 × 8 GB = 64 GB HBM).
- On a **GPU runtime** (e.g., an A100 80GB from Colab Pro+), you can enable 4-bit quantisation via `load_in_4bit=True` with bitsandbytes, which reduces the model to ~1 GB. Unsloth's `FastLanguageModel` provides this with excellent speed.
- If you switch to a GPU runtime, replace the model-loading cell with the Unsloth variant shown in the commented code blocks.

**LoRA works on both TPU and GPU** — only the quantisation part changes. The adapter itself is always in full precision.

**Recommendation:** Start with the TPU path (default below). If training is too slow or memory-constrained, switch to a Colab A100 GPU runtime and enable the Unsloth path.

In [ ]:
# ── Install dependencies ─────────────────────────────────────────────────────
# We pin versions for reproducibility. The TRL library provides SFTTrainer
# which handles the training loop cleanly.
!pip install -q \
    transformers==4.45.0 \
    datasets==2.21.0 \
    trl==0.11.4 \
    peft==0.13.2 \
    accelerate==0.34.2 \
    sacrebleu==2.4.3 \
    unbabel-comet==2.2.4 \
    torch==2.4.0 \
    evaluate==0.4.3

# For GPU runtime with Unsloth, uncomment:
# !pip install -q unsloth[colab-new] bitsandbytes>=0.43.0

print('Dependencies installed.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_DIR    = '/content/drive/MyDrive/podcast_translation'
DATA_DIR    = os.path.join(BASE_DIR, 'data')
ADAPTER_DIR = os.path.join(BASE_DIR, 'sft_adapter')
os.makedirs(ADAPTER_DIR, exist_ok=True)

print(f'Adapter will be saved to: {ADAPTER_DIR}')

In [ ]:
import json
import math
import random
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

random.seed(42)
torch.manual_seed(42)

MODEL_NAME  = 'Qwen/Qwen2.5-1.5B-Instruct'
MAX_SEQ_LEN = 256   # Generous upper bound for German segment + English translation

# Detect device
if torch.cuda.is_available():
    DEVICE = 'cuda'
elif hasattr(torch, 'xla') or 'TPU' in str(os.environ.get('COLAB_BACKEND_VERSION', '')):
    DEVICE = 'xla'
else:
    DEVICE = 'cpu'
print(f'Training device: {DEVICE}')

---
## Decision note: Prompt format

The prompt template matters enormously. It determines what the model learns to respond to, and an inconsistent template at inference time is one of the most common causes of poor fine-tuned model performance.

Our template explicitly:
1. States the task in plain English ("Translate the following German podcast transcript...")
2. Specifies the desired register ("natural English") — this is a learnable signal
3. Uses `Translation:` as the output prefix so the model knows where to start generating

**What to experiment with:**
- Adding `[PODCAST]` or `[SPOKEN]` tags if you notice the model defaulting to written register
- Wrapping the German segment in XML-style tags, e.g. `<german>...</german>`, which some models respond to more reliably
- Using the model's native chat template via `apply_chat_template()` — this ensures the special tokens (like `<|im_start|>`) are applied correctly, which can improve convergence speed

**Why not use the chat template?** We deliberately use a plain-text prompt here to keep the notebook self-contained and easy to read. If you find the model struggles to follow instructions after fine-tuning, switch to `apply_chat_template()` with `role="user"` for the German segment and `role="assistant"` for the English translation.

In [ ]:
# ── Prompt template ──────────────────────────────────────────────────────────
# This is the single most important string in the project. Keep it consistent
# across SFT, GRPO, evaluation, and inference.
PROMPT_TEMPLATE = (
    "Translate the following German podcast transcript to natural English.\n\n"
    "German: {german}\n\n"
    "Translation:"
)

def format_example(german: str, english: str) -> str:
    """Format a (German, English) pair into a single training string.
    
    The model learns to predict everything after 'Translation:',
    so the loss is computed only on the English output tokens.
    We add a space after the colon so the model starts generating
    cleanly without leading whitespace issues.
    """
    prompt = PROMPT_TEMPLATE.format(german=german)
    return prompt + ' ' + english

# Quick sanity check
example_output = format_example(
    'Das ist ein Beispiel für einen deutschen Podcast-Satz.',
    'This is an example of a German podcast sentence.'
)
print(example_output)
print(f'\nCharacters: {len(example_output)}')

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
def load_jsonl(path: str) -> list:
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

train_data = load_jsonl(os.path.join(DATA_DIR, 'train.jsonl'))
val_data   = load_jsonl(os.path.join(DATA_DIR, 'val.jsonl'))

def make_hf_dataset(data: list) -> Dataset:
    """Convert list of {de, en} dicts to a HuggingFace Dataset."""
    texts = [format_example(p['de'], p['en']) for p in data]
    return Dataset.from_dict({'text': texts})

train_ds = make_hf_dataset(train_data)
val_ds   = make_hf_dataset(val_data)

print(f'Train: {len(train_ds):,} examples')
print(f'Val:   {len(val_ds):,} examples')
print(f'\nSample formatted example:\n{train_ds[0]["text"]}')

---
## Decision note: LoRA rank and alpha

LoRA (Low-Rank Adaptation) inserts small trainable matrices into the attention layers. The two key hyperparameters are:

- **`r` (rank):** The number of dimensions in the low-rank update. Higher rank = more parameters = more expressive adapter, but also more compute and more risk of overfitting. For a *translation* task (which requires the model to preserve meaning, not learn new knowledge), a moderate rank of **16–32** is generally sufficient. We use `r=16`.
- **`lora_alpha`:** Controls the scaling of the adapter's contribution to the forward pass. The effective learning rate of the adapter is proportional to `lora_alpha / r`. Setting `alpha = 2 × r` (here, `alpha=32`) is a common default that keeps the adapter from dominating the frozen weights too early.

**What happens if you increase rank?** The adapter learns a richer update space, which helps for tasks that require more creative generation. For translation, it mainly helps if your training data covers a wide variety of topics and registers. Diminishing returns set in above rank 64 for a 1.5B model.

**What happens if you decrease rank?** The adapter is more parameter-efficient and trains faster, but may not fully adapt the model's translation style. Rank 8 is usable; rank 4 is usually too restrictive for translation.

**Target modules:** We apply LoRA to the query and value projection matrices (`q_proj`, `v_proj`) as well as the key projection and output projection. For Qwen2.5, the attention is grouped-query attention (GQA), so we also target `k_proj` and the MLP layers (`gate_proj`, `up_proj`, `down_proj`) to improve multilingual adaptation.

In [ ]:
# ── Load tokeniser and model ──────────────────────────────────────────────────
print(f'Loading {MODEL_NAME} ...')
tokeniser = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokeniser.pad_token is None:
    tokeniser.pad_token = tokeniser.eos_token

# TPU/bfloat16 path (default)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

# GPU path with Unsloth (uncomment for GPU runtime):
# from unsloth import FastLanguageModel
# model, tokeniser = FastLanguageModel.from_pretrained(
#     MODEL_NAME,
#     max_seq_length=MAX_SEQ_LEN,
#     load_in_4bit=True,
#     dtype=None,   # Auto-detect
# )

print(f'Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Apply LoRA ────────────────────────────────────────────────────────────────
LORA_R      = 16
LORA_ALPHA  = 32
LORA_DROPOUT = 0.05

# Qwen2.5 attention module names:
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

# For Unsloth GPU path, replace the above with:
# model = FastLanguageModel.get_peft_model(
#     model, r=LORA_R, lora_alpha=LORA_ALPHA,
#     target_modules=LORA_TARGET_MODULES,
#     lora_dropout=LORA_DROPOUT, bias='none', use_gradient_checkpointing='unsloth'
# )

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected: ~0.5–1% of total parameters are trainable — this is the point of LoRA.

---
## Decision note: Learning rate schedule

We use a **cosine schedule with linear warmup** for several reasons:

1. **Warmup** (typically 5–10% of total steps): The LoRA adapter is randomly initialised. If we immediately apply the full learning rate, the large initial gradients can destabilise the frozen base model's representations. A linear warmup ramps the learning rate from 0 to its maximum, giving the adapter time to find a sensible starting direction.

2. **Cosine decay:** After the peak, the learning rate decays smoothly to near-zero following a cosine curve. Compared to a linear decay, cosine decay keeps the learning rate higher for longer in the middle of training (allowing broad exploration) and then decays more aggressively near the end (allowing fine-grained convergence). This is particularly beneficial for translation where the model needs both to learn new vocabulary patterns early and to fine-tune subtle phrasing choices late.

3. **Why not constant LR?** A constant learning rate tends to overshoot the minimum near the end of training, producing a noisy loss curve and slightly worse final performance.

**What overfitting looks like in SFT for translation:**
- Training loss continues to fall while validation loss plateaus or rises
- Validation chrF++ and COMET stop improving or begin to decline
- The model starts producing translations that sound memorised — very fluent but oddly phrased in ways that mirror specific training examples

**When to stop:** Use the validation COMET score as the primary stopping criterion. If COMET has not improved by more than 0.5 points over the last 2 evaluation checkpoints, stop. This is implemented via `load_best_model_at_end=True` below.

In [ ]:
# ── Training configuration ────────────────────────────────────────────────────
# The batch size and gradient accumulation are tuned for a TPU v2-8.
# On a single A100 GPU, you might use per_device_batch_size=4 and
# gradient_accumulation_steps=4 for an equivalent effective batch size of 16.

LEARNING_RATE       = 2e-4
NUM_EPOCHS          = 3
WARMUP_RATIO        = 0.06    # ~6% of steps used for linear warmup
EVAL_STEPS          = 500     # Evaluate every 500 steps
SAVE_STEPS          = 500
PER_DEVICE_BATCH    = 8
GRAD_ACCUM_STEPS    = 2       # Effective batch = 8 × 2 = 16

sft_config = SFTConfig(
    output_dir='/content/sft_checkpoints',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type='cosine',
    warmup_ratio=WARMUP_RATIO,
    eval_strategy='steps',
    eval_steps=EVAL_STEPS,
    save_strategy='steps',
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_dir='/content/sft_logs',
    logging_steps=50,
    bf16=True,          # Native bfloat16 for TPU; switch to fp16=True on GPU
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    report_to='none',   # Set to 'wandb' if you have a W&B account
    seed=42,
)

In [ ]:
# ── Evaluation callback: chrF++ and COMET ─────────────────────────────────────
# We compute chrF++ at every eval step (fast, no GPU needed) and COMET
# less frequently (requires a separate model download) to keep training fast.
# Results are printed to stdout and logged so you can see the learning curve.
import sacrebleu
from transformers import TrainerCallback, TrainerState, TrainerControl

CHRF_EVAL_PAIRS = val_data[:200]   # Use first 200 val pairs for fast chrF++ eval
CHRF_HISTORY    = []

class TranslationEvalCallback(TrainerCallback):
    """Compute chrF++ on a small validation sample at each eval step.
    
    COMET evaluation is more expensive and is run manually after training
    using the full validation set in the evaluation notebook.
    """
    def on_evaluate(self, args, state: TrainerState, control: TrainerControl,
                    model=None, tokenizer=None, **kwargs):
        if model is None or tokenizer is None:
            return
        model.eval()
        hypotheses = []
        references  = []
        with torch.no_grad():
            for pair in CHRF_EVAL_PAIRS[:50]:   # 50 pairs for speed
                prompt = PROMPT_TEMPLATE.format(german=pair['de']) + ' '
                inputs = tokenizer(
                    prompt,
                    return_tensors='pt',
                    truncation=True,
                    max_length=MAX_SEQ_LEN
                ).to(model.device)
                out = model.generate(
                    **inputs,
                    max_new_tokens=150,
                    do_sample=False,
                    pad_token_id=tokenizer.eos_token_id
                )
                generated = tokenizer.decode(
                    out[0][inputs['input_ids'].shape[-1]:],
                    skip_special_tokens=True
                ).strip()
                hypotheses.append(generated)
                references.append(pair['en'])

        chrf = sacrebleu.corpus_chrf(hypotheses, [references])
        score = chrf.score
        CHRF_HISTORY.append({'step': state.global_step, 'chrf': score})
        print(f'\n[Step {state.global_step}] chrF++ = {score:.2f}')
        model.train()

print('Evaluation callback defined.')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    callbacks=[TranslationEvalCallback()],
)

print('Starting SFT training...')
trainer.train()
print('Training complete.')

In [ ]:
# ── Plot learning curves ──────────────────────────────────────────────────────
import matplotlib.pyplot as plt

log_history = trainer.state.log_history
train_losses = [(e['step'], e['loss']) for e in log_history if 'loss' in e]
eval_losses  = [(e['step'], e['eval_loss']) for e in log_history if 'eval_loss' in e]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

if train_losses:
    steps, losses = zip(*train_losses)
    axes[0].plot(steps, losses, label='Train loss', color='steelblue')
if eval_losses:
    steps, losses = zip(*eval_losses)
    axes[0].plot(steps, losses, label='Val loss', color='coral')
axes[0].set_title('Cross-entropy loss')
axes[0].set_xlabel('Step')
axes[0].legend()

if CHRF_HISTORY:
    steps  = [x['step'] for x in CHRF_HISTORY]
    chrfs  = [x['chrf']  for x in CHRF_HISTORY]
    axes[1].plot(steps, chrfs, color='mediumseagreen', marker='o')
    axes[1].set_title('chrF++ on validation sample')
    axes[1].set_xlabel('Step')
    axes[1].set_ylabel('chrF++')

plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, 'sft_learning_curves.png'), dpi=150)
plt.show()
print('Learning curve saved.')

In [ ]:
# ── Save LoRA adapter to Google Drive ────────────────────────────────────────
# We save only the adapter (not the full model) to save Drive space.
# The base model can be re-downloaded at any time; only the adapter
# weights are the product of our training effort.
model.save_pretrained(ADAPTER_DIR)
tokeniser.save_pretrained(ADAPTER_DIR)
print(f'LoRA adapter saved to {ADAPTER_DIR}')
print('Files:', os.listdir(ADAPTER_DIR))

In [ ]:
# ── Quick qualitative test ────────────────────────────────────────────────────
# Translate a few held-out examples to get a feel for the output quality.
# Do NOT use the sacred test set here.
model.eval()
test_sentences = [
    'Und ich glaube, das ist wirklich der Kern der Sache, über die wir heute sprechen wollen.',
    'Ja, genau, das sehe ich auch so, und deswegen finde ich das so wichtig.',
    'Also wenn wir uns die Zahlen anschauen, dann sehen wir eigentlich ein klares Bild.',
]

for de in test_sentences:
    prompt = PROMPT_TEMPLATE.format(german=de) + ' '
    inputs = tokeniser(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=100, do_sample=False,
            pad_token_id=tokeniser.eos_token_id
        )
    en = tokeniser.decode(
        out[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True
    ).strip()
    print(f'DE: {de}')
    print(f'EN: {en}')
    print()

---
## Summary and Next Steps

After running this notebook you should have:
- A saved LoRA adapter in `podcast_translation/sft_adapter/` on your Drive
- A learning curve plot showing both train and validation loss alongside chrF++ scores
- A rough sense of the translation quality from the qualitative test above

**Interpreting the learning curves:**
- If train and val loss track each other closely: healthy. Continue to GRPO.
- If val loss plateaus but train loss keeps falling: overfitting. You stopped at the right checkpoint (thanks to `load_best_model_at_end=True`), but consider reducing `NUM_EPOCHS` for future runs.
- If chrF++ is flat from the start: check that the prompt template is consistent and that the data was loaded correctly.

**Proceed to: `03_rl_grpo_training.ipynb`**